# 容量约束设施选址问题(CFLP)

**类别：** 选址

来源：[https://www.hexaly.com/templates/capacitated-facility-location-problem-cflp](https://www.hexaly.com/templates/capacitated-facility-location-problem-cflp)


## 问题描述

**在容量约束设施选址问题(Capacitated Facility Location Problem, CFLP)**中,若干具有已知需求的客户点必须被分配给可用的设施。分配给每个设施的客户点总需求不得超过其容量。每个设施都有一个固定的开启费用,只要该设施至少服务一个客户点就必须支付。目标是最小化总成本,该成本为所有已开启设施的开启费用与所有客户点的分配费用之和。

### 学习要点

- 添加 set decision variables 以建模分配给每个设施的客户点
- 使用 lambda 表达式 计算每个设施的总需求与总费用


## 数据

所提供的数据文件来自 [OR-LIB](http://people.brunel.ac.uk/~mastjjb/jeb/orlib/pmedcapinfo.html)。数据格式如下:

- 第一行:候选设施数量与客户点数量
- 每个设施的容量与开启费用
- 每个客户点的需求
- 每个设施与每个客户点之间的分配费用


## 建模方法

容量约束设施选址问题(CFLP)的 OptAgent 模型使用 set decision variables 来表示分配给每个设施的客户点。借助 partition 算子,我们确保每个客户点恰好被分配到一个设施。

我们可以使用需求数组上的 at 算子来访问序列中每个客户点的需求。每个设施所服务的总需求通过一个 lambda 函数 计算,该函数将 sum 算子应用于所有关联客户点的需求。该总需求必须不超过该设施的容量。

类似地,我们将每个设施的分配费用计算为其所服务的所有客户点的分配价格之和。使用 count 算子,我们检查每个设施是否至少服务一个客户点。如果是,则还需支付该设施的开启费用。

目标函数为所有设施的开启费用与分配费用之和。


## Python 实现


In [1]:
from pathlib import Path

from optagent import ModelBuilder, solve


def read_data(filename):
    """Read a CFLP instance in the OR-LIB format."""
    with open(filename, encoding="utf-8") as f:
        tokens = f.read().split()
    it = iter(tokens)
    nb_max_facilities = int(next(it))
    nb_sites = int(next(it))
    capacity_data = []
    opening_price_data = []
    for _ in range(nb_max_facilities):
        capacity_data.append(float(next(it)))
        opening_price_data.append(float(next(it)))
    demand_data = [float(next(it)) for _ in range(nb_sites)]
    allocation_price_data = [
        [float(next(it)) for _ in range(nb_sites)] for _ in range(nb_max_facilities)
    ]
    return nb_max_facilities, nb_sites, capacity_data, opening_price_data, demand_data, allocation_price_data


def solve_instance(instance_file, output_file=None, time_limit=10):
    nb_max_facilities, nb_sites, capacity_data, opening_price_data, demand_data, allocation_price_data = read_data(instance_file)

    model = ModelBuilder()
    # Each facility assignment is the set of sites served by that facility.
    facility_assignments = [model.set(nb_sites) for _ in range(nb_max_facilities)]
    # Every site is served by exactly one facility.
    model.constraint(model.partition(facility_assignments))

    demand = model.array(demand_data)
    allocation_price = model.array(allocation_price_data)

    cost = []
    for f in range(nb_max_facilities):
        facility = facility_assignments[f]
        size = model.count(facility)

        # Capacity constraint: total demand served by this facility.
        demand_lambda = model.lambda_function(lambda i: demand[i])
        model.constraint(model.sum(facility, demand_lambda) <= capacity_data[f])

        # Allocation cost + opening cost when the facility serves at least one site.
        cost_selector = model.lambda_function(lambda i, f=f: model.at(allocation_price, f, i))
        cost.append(model.sum(facility, cost_selector) + opening_price_data[f] * (size > 0))

    total_cost = model.sum(*cost)
    model.minimize(total_cost, name="total_cost")

    solution = solve(model, time_limit_s=float(time_limit))
    lines = [
        f"Facilities = {nb_max_facilities}; Sites = {nb_sites}; "
        f"Total cost = {solution.objective_value}; Status = {solution.status.value}"
    ]
    open_facilities = 0
    for f in range(nb_max_facilities):
        sites = solution.variable_values[facility_assignments[f].node_id]
        if sites:
            open_facilities += 1
            lines.append(f"Facility {f} -> sites {sorted(sites)}")
    lines.append(f"Open facilities: {open_facilities}/{nb_max_facilities}")
    result_text = "\n".join(lines)
    print(result_text)
    if output_file is not None:
        Path(output_file).write_text(result_text + "\n", encoding="utf-8")
    return solution


def main(input_file, output_file=None, time_limit=10):
    return solve_instance(input_file, output_file, time_limit)


In [2]:
INSTANCE_DIR = Path.cwd() / "instances"
print("Instances:", INSTANCE_DIR)


Instances: /Users/dongbox/work/opt-agent/examples/examples/hexaly/capacitated_facility_location_problem_cflp/instances


In [3]:
solution_cap61 = main(
    INSTANCE_DIR / "cap61",
    time_limit=1,
)


Starting OptAgent PORTFOLIO
Parameters: time_limit=1s threads=auto seed=0
Solve summary:
  status: FEASIBLE
  objective: 1.86713e+06
  improvements: initial=1 search=3
  evaluated: 80
  wall_time: 1.0104s
  termination: wall_time_exhausted


Facilities = 16; Sites = 50; Total cost = 1867133.375; Status = feasible
Facility 0 -> sites [20, 25, 33, 40]
Facility 1 -> sites [10, 26, 36, 46]
Facility 2 -> sites [0, 3, 6, 7, 17, 37, 41, 44, 48]
Facility 3 -> sites [1, 2, 4, 5, 8, 9, 11, 12, 13, 14, 15, 16, 18, 19, 21, 22, 23, 24, 27, 28, 29, 30, 31, 32, 34, 35, 38, 39, 42, 43, 45, 47, 49]
Open facilities: 4/16
